# NB05: Environmental Distribution of AMR Genes

**Goal**: Test H4 — do species from different environments carry different AMR profiles? Use NCBI environment metadata and AlphaEarth embeddings.

**Caveats**: 
- `ncbi_env` is EAV-format (key-value), sparse and inconsistent
- AlphaEarth embeddings cover only 28% of genomes
- Strong clinical/pathogen sampling bias in genome databases

**Depends on**: NB01 outputs + Spark session

**Outputs**:
- `../data/amr_by_environment.csv`
- `../figures/amr_environment_*.png`

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

try:
    spark = get_spark_session()
except NameError:
    sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..', 'scripts'))
    from get_spark_session import get_spark_session
    spark = get_spark_session()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')

plt.rcParams.update({
    'figure.figsize': (12, 6), 'figure.dpi': 150, 'font.size': 11,
    'axes.titlesize': 13, 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})

amr = pd.read_csv(DATA_DIR / 'amr_census.csv')
species = pd.read_csv(DATA_DIR / 'amr_species_summary.csv')
print(f"Loaded {len(amr):,} AMR clusters, {len(species):,} species")

## 1. Environment Classification from NCBI Metadata

Extract isolation_source from `ncbi_env` and classify into broad environment categories. This is EAV-format data — we need to pivot and harmonize.

In [ ]:
# Get isolation_source for genomes in AMR-carrying species
# First, get the species list
amr_species = species['gtdb_species_clade_id'].unique().tolist()
spark.createDataFrame([(x,) for x in amr_species], ['gtdb_species_clade_id']).createOrReplaceTempView('amr_species')

# Get genome IDs for AMR species
# Then get isolation_source from ncbi_env
env_data = spark.sql("""
    SELECT g.gtdb_species_clade_id, g.genome_id, e.`key`, e.`value`
    FROM kbase_ke_pangenome.genome g
    JOIN amr_species sp ON g.gtdb_species_clade_id = sp.gtdb_species_clade_id
    JOIN kbase_ke_pangenome.ncbi_env e ON g.genome_id = e.accession
    WHERE e.`key` IN ('isolation_source', 'host', 'env_broad_scale', 'env_local_scale')
""").toPandas()

print(f"Environment metadata records: {len(env_data):,}")
print(f"Genomes with env data: {env_data['genome_id'].nunique():,}")
print(f"Species with env data: {env_data['gtdb_species_clade_id'].nunique():,}")
print(f"\nMetadata keys:")
print(env_data['key'].value_counts().to_string())

In [ ]:
# Classify isolation_source into broad categories
# Reuse the harmonization approach from env_embedding_explorer
def classify_environment(source):
    """Classify isolation_source text into broad categories."""
    if pd.isna(source):
        return 'Unknown'
    s = str(source).lower()
    
    # Clinical/Human
    if any(k in s for k in ['blood', 'sputum', 'urine', 'wound', 'clinical', 'patient',
                             'hospital', 'abscess', 'csf', 'biopsy', 'stool', 'feces',
                             'human', 'homo sapiens', 'nasopharyn', 'throat', 'skin',
                             'respiratory', 'lung', 'rectal', 'vaginal', 'urinary']):
        return 'Human/Clinical'
    
    # Animal host
    if any(k in s for k in ['chicken', 'cattle', 'pig', 'swine', 'bovine', 'poultry',
                             'dog', 'cat', 'fish', 'mouse', 'rat', 'animal', 'veterinary',
                             'insect', 'tick', 'mosquito', 'bird']):
        return 'Animal'
    
    # Food
    if any(k in s for k in ['food', 'milk', 'cheese', 'meat', 'ferment', 'yogurt',
                             'kimchi', 'sausage', 'vegetable', 'fruit', 'grain']):
        return 'Food'
    
    # Soil/Terrestrial
    if any(k in s for k in ['soil', 'rhizosphere', 'root', 'sediment', 'mud', 'compost',
                             'agricultural', 'farm', 'forest', 'grassland']):
        return 'Soil/Terrestrial'
    
    # Aquatic
    if any(k in s for k in ['water', 'river', 'lake', 'ocean', 'sea', 'marine', 'aquatic',
                             'wastewater', 'sewage', 'freshwater', 'groundwater']):
        return 'Aquatic'
    
    # Plant
    if any(k in s for k in ['plant', 'leaf', 'stem', 'flower', 'seed', 'phyllosphere']):
        return 'Plant'
    
    return 'Other/Unknown'

# Apply to isolation_source
iso = env_data[env_data['key'] == 'isolation_source'].copy()
iso['env_category'] = iso['value'].apply(classify_environment)

# Species-level: majority environment classification
species_env = iso.groupby('gtdb_species_clade_id')['env_category'].agg(
    lambda x: x.value_counts().index[0]  # mode
).reset_index()
species_env.columns = ['gtdb_species_clade_id', 'primary_env']

print(f"\n=== Environment Classification ===")
print(iso['env_category'].value_counts().to_string())
print(f"\nSpecies with environment classification: {len(species_env):,}")

In [ ]:
# Merge environment with species AMR data
species_with_env = species.merge(species_env, on='gtdb_species_clade_id', how='inner')
species_with_env = species_with_env[species_with_env['primary_env'] != 'Other/Unknown']

print(f"Species with AMR + environment: {len(species_with_env):,}")
print(f"\n=== AMR by Environment ===")

env_amr = species_with_env.groupby('primary_env').agg(
    n_species=('gtdb_species_clade_id', 'count'),
    mean_amr=('n_amr', 'mean'),
    median_amr=('n_amr', 'median'),
    total_amr=('n_amr', 'sum'),
    mean_core_pct=('pct_core_amr', 'mean')
).round(2).sort_values('mean_amr', ascending=False)

print(env_amr.to_string())

# Kruskal-Wallis test: does AMR count differ by environment?
groups = [g['n_amr'].values for _, g in species_with_env.groupby('primary_env')]
h_stat, p_kw = stats.kruskal(*[g for g in groups if len(g) >= 5])
print(f"\nKruskal-Wallis (AMR count ~ environment): H={h_stat:.1f}, p={p_kw:.2e}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot: AMR count by environment
env_order = env_amr.index.tolist()
bp_data = [species_with_env[species_with_env['primary_env'] == e]['n_amr'].values for e in env_order]
bp = axes[0].boxplot(bp_data, labels=env_order, vert=True, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#e74c3c')
    patch.set_alpha(0.6)
axes[0].set_ylabel('AMR Clusters per Species')
axes[0].set_title('AMR Gene Count by Isolation Environment')
axes[0].tick_params(axis='x', rotation=45)

# Bar: mean AMR by environment
axes[1].barh(env_amr.index, env_amr['mean_amr'], color='#e74c3c')
axes[1].set_xlabel('Mean AMR Clusters per Species')
axes[1].set_title('Mean AMR Density by Environment')
for i, (idx, row) in enumerate(env_amr.iterrows()):
    axes[1].text(row['mean_amr'] + 0.1, i, f'n={int(row["n_species"])}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'amr_by_environment.png')
plt.show()

species_with_env.to_csv(DATA_DIR / 'amr_by_environment.csv', index=False)
print(f"\nSaved to data/amr_by_environment.csv")